# LLM AS A JUDGE 

In [ ]:
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q      
!pip install torch sacrebleu "unbabel-comet>=2.2.0" bert-score evaluate tensorflow numpy nltk -q
!pip install git+https://github.com/google-research/bleurt.git -q

In [ ]:
import torch, datetime, re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import sacrebleu                                 
from sacrebleu.metrics import TER                 
from nltk.translate.meteor_score import single_meteor_score  
import numpy as np          
from typing import Union, Dict, List
import warnings

In [ ]:
print("Loading Prometheus (4‑bit) …")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Unbabel/M-Prometheus-7B",
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    "Unbabel/M-Prometheus-7B",
    trust_remote_code=True,
)

model.eval()

In [ ]:
templates = {
        "general":'''###Task Description:
An instruction, a response to evaluate, a **gold translation** (reference answer) and a score rubric representing an evaluation criterion are given.

1. Write a detailed feedback of maximum 100 words that assesses the response strictly according to the score rubric (translation quality), **not evaluating in general**.
2. After writing the feedback, write a score that is an integer between 1 and 5. You must refer to the rubric.
3. The output format must be EXACTLY:
"<your feedback here>
[RESULT] <integer 1-5>"
4. Do not generate any headers, closings, or extra commentary.

###The instruction to evaluate:
Traduci il seguente brano di italiano arcaico nel moderno italiano:
«{ARCHAIC_PASSAGE}»

###Response to evaluate (model output):
{MODEL_TRANSLATION}

###Gold Translation (reference answer):
{GOLD_TRANSLATION}

###Score Rubric (Translation Quality):
5 – Excellent: The translation is fluid, precise, and natural. It preserves all the semantic nuances of the original text. The syntax is modern yet respectful of the source content. The text is perfectly readable and does not feel machine‑generated. Archaic terms are rendered with accurate modern equivalents, even in ambiguous cases. Indistinguishable from a translation by a highly qualified human.
4 – Good: A generally correct and readable translation. The text appears modern and understandable, although some expressions come across as slightly mechanical or overly literal. It preserves the overall meaning well but may lose subtle cultural or historical nuances. Some terms are adapted with a good but not optimal choice. It may resemble a solid human translation, but not an outstanding one.
3 – Sufficient: An understandable translation but with evident mechanical traits or stylistically poor passages. There may be rigid constructions, syntactic calques, or unnatural lexical choices. The global meaning is retained, but the expressive richness of the original medieval text is lost. It reads like an automatic translation with minimal human supervision.
2 – Insufficient: A discontinuous or incoherent translation. Sentences are disconnected, modern vocabulary is misused, and ambiguous or obsolete words are misinterpreted. Some passages are incomprehensible or misleading. It clearly feels like a text translated automatically without human intervention. Not reliable for reading or academic study.
1 – Gravely Insufficient: A failed translation. Sentences are nonsensical, terms are mistranslated or omitted entirely. There is no grammatical or semantic coherence. The text is unusable and devoid of communicative value. Neither the original content nor its purpose can be discerned. It seems produced by a crude or faulty automatic system, utterly lacking linguistic or historical understanding.

###Feedback:''',
        "grammatic":'''###Task Description:
An instruction, a response to evaluate, a **gold translation** (reference answer) and a score rubric representing an evaluation criterion are given.

1. Write a detailed feedback of maximum 100 words that assesses the response strictly according to the score rubric (Accuracy in grammar, syntax, and conformity to modern Italian norms), **not evaluating in general**.
2. After writing the feedback, write a score that is an integer between 1 and 5. You must refer to the rubric.
3. The output format must be EXACTLY:
"<your feedback here>
[RESULT] <integer 1-5>"
4. Do not generate any headers, closings, or extra commentary.

###The instruction to evaluate:
Traduci il seguente brano di italiano arcaico nel moderno italiano:
«{ARCHAIC_PASSAGE}»

###Response to evaluate (model output):
{MODEL_TRANSLATION}

###Gold Translation (reference answer):
{GOLD_TRANSLATION}

###Score Rubric (Grammatic Quality):
5 – Flawless: Grammar, punctuation, and orthography are impeccable. Sentences flow naturally, with varied structure and perfect agreement of tense, number, and gender. Word order feels idiomatic and clear. No editing is required; the text reads like it was crafted by a seasoned native writer.
4 – Very Good: Minor, hard-to-spot issues (e.g., an omitted comma, a rare tense slip) appear only sporadically and do not impede comprehension or stylistic elegance. Sentence patterns are mostly natural, though an occasional phrasing may sound slightly mechanical.
3 – Adequate: Noticeable but non-disruptive grammatical lapses (awkward word order, repetitive structures, or several punctuation glitches). The text remains readable, yet would benefit from copy-editing to achieve professional polish.
2 – Poor: Frequent grammatical errors (verb-tense mismatches, agreement problems, run-ons or fragments) distract the reader and occasionally obscure meaning. The text feels clearly unedited and requires substantial revision to meet academic or publishing standards.
1 – Unacceptable: Pervasive, severe grammatical faults make the text hard or impossible to follow. Sentences are often ungrammatical or incomplete, punctuation is chaotic, and basic rules of syntax are ignored. The writing fails to convey information reliably.

###Feedback:''',
        "semantic":'''###Task Description:
An instruction, a response to evaluate, a **gold translation** (reference answer) and a score rubric (Fidelity to the meaning, tone, and subtle nuances of the archaic source) representing an evaluation criterion are given.

###The instruction to evaluate:
Traduci il seguente brano di italiano arcaico nel moderno italiano:
«{ARCHAIC_PASSAGE}»

###Response to evaluate (model output):
{MODEL_TRANSLATION}

###Gold Translation (reference answer):
{GOLD_TRANSLATION}

###Score Rubric (Semantic Quality):
5 – Excellent: All intended meanings, nuances, and relationships are conveyed with precision. Concepts are accurately represented, contextually appropriate, and internally consistent. No misinterpretations or omissions; the text mirrors the source’s full informational depth.
4 – Good: The core meaning and almost all subtleties are preserved. A few minor shades of sense or cultural references may be softened, but factual content and logical relations remain intact. Overall message fidelity is high.
3 – Sufficient: Central ideas are captured, yet several nuances, implications, or culturally loaded elements are diluted or missed. Some statements may feel slightly out of context, but the general message survives without critical distortion.
2 – Insufficient: Major semantic inaccuracies, omissions, or additions alter the author’s intent. Key concepts are confused or simplified, leading to a partial or skewed understanding of the original content.
1 – Gravely Insufficient: Meaning is largely lost or contradicted. Essential ideas are misrepresented, context is ignored, and the text may introduce content not present in the source. Readers cannot rely on the translation to grasp the original message.

###Feedback:'''}
def judge(to_predict, prediction, gold, mode, verbose=0):
    if mode not in set(templates.keys()): raise ValueError("unaccepeted mode")
    
    prompt = templates[mode].format(
        ARCHAIC_PASSAGE=to_predict.strip(),
        MODEL_TRANSLATION=prediction.strip(),
        GOLD_TRANSLATION=gold.strip()
    )

    # 2️⃣  Run Prometheus 2 --------------------------
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    lunghezza_input = len(inputs[0])
    if verbose>0:
        print(f"ORIGINAL:\n {to_predict}")
        print(f"PREDICTION:\n{prediction} ")
        print(f"GOLD:\n {gold}")
        print("▶ Generating judgement ...")
        t0 = datetime.datetime.now()

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens= 256, do_sample=False)

    raw = tokenizer.decode(out[0][lunghezza_input:], skip_special_tokens=True)
    

    if verbose>1:
        print("RAW OUTPUT OF PROMETHEUS:")
        print(raw)
        print("END RAW")

    # 3️⃣  Parse feedback + numeric score ------------
    m = re.search(
        r"^(.*?)\s*\[RESULT\]\s*([1-5])\b",   # ① feedback   ② score
        raw,
        re.S                                   # dot matches newlines
    )
    if not m:
        print("!!!!!!  ERROR: Cannot parse Prometheus output:\n" + raw)
        print(f"Skipping the decomposition of judjment of the sentence: {prediction}")
        feedback = raw
        score    = raw
    else:
        feedback = m.group(1).strip()
        score    = int(m.group(2))
        if verbose>0:
            print(f"SCORE: {score}")
        if verbose>1:
            print(f"FEEDBACK: {feedback}")
            print("⏱", datetime.datetime.now() - t0)

    return {"feedback": feedback, "score": score, "raw": raw}

In [ ]:
# This is a small trick to avoid reloading models on repeated calls in the same script.
_MODEL_CACHE = {}

def evaluate_metrics(
    original: Union[str, List[str]],
    gold: Union[str, List[str]],
    predicted: Union[str, List[str]]
) -> Union[Dict[str, float], Dict[str, Dict]]:
    """
    A simple, all-in-one function to evaluate MT metrics.

    Handles both single sentences (str) and corpora (List[str]).
    It is resilient to failures in individual metric calculations.

    - For a single sentence, returns a flat dictionary of scores.
    - For a corpus, returns a dictionary where sentence-level metrics
      (BERTScore, BLEURT, COMET-22) contain a 'scores' list with a
      score for each sentence.
    """
    is_corpus = isinstance(gold, list)

    # --- 1. Standardize inputs to lists for uniform processing ---
    if not is_corpus:
        originals, golds, predicteds = [original], [gold], [predicted]
    else:
        if not (isinstance(original, list) and isinstance(predicted, list)):
            raise ValueError("If one input is a list (corpus), all must be lists.")
        originals, golds, predicteds = original, gold, predicted
    
    results = {}
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Running evaluation on {len(golds)} sentence(s) using device: {device}")

    # --- 2. Sacrebleu-based Metrics (fast and reliable) ---
    results['BLEU'] = sacrebleu.corpus_bleu(predicteds, [golds]).score / 100
    results['chrF++'] = sacrebleu.corpus_chrf(predicteds, [golds], word_order=2).score / 100
    ter_score = sacrebleu.corpus_ter(predicteds, [golds]).score / 100
    results['TER (inv)'] = 1.0 - ter_score

    # --- 3. Model-based Metrics (slower, wrapped for safety) ---
    
    # BERTScore
    try:
        if 'bert_scorer' not in _MODEL_CACHE:
            print("Loading BERTScore model...")
            from bert_score import BERTScorer
            _MODEL_CACHE['bert_scorer'] = BERTScorer(lang="it", model_type='bert-base-multilingual-cased', device=device)
        _, _, F1 = _MODEL_CACHE['bert_scorer'].score(predicteds, golds, batch_size=16)
        results['BERTScore'] = F1.tolist() # <-- CHANGED: Store the list of scores
    except Exception as e:
        warnings.warn(f"Could not calculate BERTScore: {e}")
        results['BERTScore'] = [-1.0] * len(golds)

    # BLEURT
    try:
        if 'bleurt_scorer' not in _MODEL_CACHE:
            print("Loading BLEURT-20 model...")
            from evaluate import load as hf_load
            _MODEL_CACHE['bleurt_scorer'] = hf_load("bleurt", "BLEURT-20", module_type="metric")
        bleurt_scores = _MODEL_CACHE['bleurt_scorer'].compute(predictions=predicteds, references=golds)['scores']
        results['BLEURT'] = bleurt_scores # <-- CHANGED: Store the list of scores
    except Exception as e:
        warnings.warn(f"Could not calculate BLEURT. Error: {e}")
        results['BLEURT'] = [-1.0] * len(golds)

    # COMET
    try:
        if 'comet_model' not in _MODEL_CACHE:
            print("Loading COMET-22 model...")
            import comet
            model_path = comet.download_model("Unbabel/wmt22-comet-da")
            _MODEL_CACHE['comet_model'] = comet.load_from_checkpoint(model_path)
        comet_data = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(originals, predicteds, golds)]
        comet_output = _MODEL_CACHE['comet_model'].predict(comet_data, batch_size=8, gpus=1, progress_bar=False)
        results['COMET-22'] = comet_output.scores # <-- CHANGED: Store the list of scores
    except Exception as e:
        warnings.warn(f"Could not calculate COMET-22. Error: {e}")
        results['COMET-22'] = [-1.0] * len(golds)

    # --- 4. Format Output ---
    if not is_corpus:
        # For a single sentence, extract the single score from the list
        return {
            'BLEU': round(results['BLEU'], 4),
            'chrF++': round(results['chrF++'], 4),
            'TER (inv)': round(results['TER (inv)'], 4),
            'BERTScore': round(results['BERTScore'][0], 4),
            'BLEURT': round(results['BLEURT'][0], 4),
            'COMET-22': round(results['COMET-22'][0], 4),
        }
    else:
        # For a corpus, return corpus scores and lists of sentence scores
        return {
            'BLEU': {'corpus_score': round(results.get('BLEU', -1), 4)},
            'chrF++': {'corpus_score': round(results.get('chrF++', -1), 4)},
            'TER (inv)': {'corpus_score': round(results.get('TER (inv)', -1), 4)},
            'BERTScore': {'scores': [round(s, 4) for s in results.get('BERTScore', [])]},
            'BLEURT': {'scores': [round(s, 4) for s in results.get('BLEURT', [])]},
            'COMET-22': {'scores': [round(s, 4) for s in results.get('COMET-22', [])]},
        }

In [ ]:
# read the dataset with the llm traslations


translated_dataset = pd.read_csv("/kaggle/input/popuss/predicted_opus.csv")

In [ ]:
import os
import pandas as pd

file_paths = [
    "/kaggle/input/predicted/predicted_mistral.xlsx",
    "/kaggle/input/predicted/predicted_opus.csv"
]
modes = ["grammatic", "semantic"]
for mode in modes:
    for path in file_paths:
        if path.endswith(".xlsx"):
            translated_dataset = pd.read_excel(path)
            translated_dataset = translated_dataset.rename(columns={"Mistral Traslation":"prediction"})
        elif path.endswith(".csv"):
            translated_dataset = pd.read_csv(path)
        else:
            raise ValueError(f"Unsupported file type: {path}")
    
        # Judging
        scores = []
        feedbacks = []
        print(f"\n\nJudging {path}, {mode}\n\n")
        for idx, row in translated_dataset.iterrows():
            to_predict = row["Sentence"]
            gold      = row["TRADUZIONE ITALIANA"]
            prediction = row["prediction"]
    
            judgement = judge(
                to_predict,
                prediction,
                gold,
                mode=mode,
                verbose=1
            )
            scores.append(judgement["score"])
            feedbacks.append(judgement["feedback"])
        
        # e.g. inspect or save
        mt_metrics = evaluate_metrics(
            translated_dataset["Sentence"].to_list(),
            translated_dataset["TRADUZIONE ITALIANA"].to_list(),
            translated_dataset["prediction"].to_list()
        )
        translated_dataset['Score']    = scores
        translated_dataset['Feedback'] = feedbacks
        translated_dataset['BERTScore'] = mt_metrics['BERTScore']['scores']
        translated_dataset['COMET-22']  = mt_metrics['COMET-22']['scores']
        translated_dataset['BLEURT']    = mt_metrics['BLEURT']['scores']
        translated_dataset['BLEU']      = pd.Series(mt_metrics['BLEU']['corpus_score'], index=translated_dataset.index[[0]])
        translated_dataset['chrF++']    = pd.Series(mt_metrics['chrF++']['corpus_score'], index=translated_dataset.index[[0]])
        translated_dataset['TER (inv)']= pd.Series(mt_metrics['TER (inv)']['corpus_score'], index=translated_dataset.index[[0]])
    
        # --- DYNAMIC OUTPUT PATH ---
        base = os.path.splitext(os.path.basename(path))[0]              # e.g. "predicted_mistral"
        output_path = f"/kaggle/working/{base}_traduzioni_con_score.xlsx"  # e.g. "predicted_mistral_traduzioni_con_score.xlsx"
        translated_dataset.to_excel(output_path, index=False)
    
        print(f"\nResults for {path} (saved to {output_path}):")
        print(translated_dataset[["Sentence", "prediction", "score"]].head())
        print("\n\n\n\n\n\n")


# # Cohen Kappa Coeffincent Cohen Kappa Coeffincent 

In [ ]:
# Kappa coefficent calculus

import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np

df = pd.read_excel("/kaggle/working/mistral_traduzioni_con_score.xlsx") 

# Aggiunge una colonna "Human Score" con valori interi random da 1 a 5
df['Human Score'] = np.random.randint(1, 6, size=len(df))

print(df['Human Score'])

In [ ]:

# Verifica che i valori siano da 1 a 5
assert df["Human Score"].between(1, 5).all(), "Valori fuori range in 'valutazione_umana'"
assert df["Score"].between(1, 5).all(), "Valori fuori range in 'valutazione_llm'"

# Calcolo del coefficiente di Cohen's Kappa
kappa = cohen_kappa_score(df["Human Score"], df["Score"])

print(f"Cohen's Kappa: {kappa:.4f}")

In [ ]:
# Confuzion Matrix

cm = confusion_matrix(df["Human Score"], df["Score"], labels=[1, 2, 3, 4, 5])

# Plot con seaborn
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5])
plt.xlabel("Score")
plt.ylabel("Human Score")
plt.title("Matrice di Confusione")
plt.show()


In [ ]:
# WEIGHTED KAPPA QUADRATIC

from sklearn.metrics import cohen_kappa_score

# Pesi lineari: penalizzano proporzionalmente
kappa_linear = cohen_kappa_score(df["Human Score"], df["Score"], weights="linear")

# Pesi quadratici: penalizzano fortemente i disaccordi ampi
kappa_quadratic = cohen_kappa_score(df["Human Score"], df["Score"], weights="quadratic")

print(f"Cohen's Kappa (Linear): {kappa_linear:.4f}")
print(f"Cohen's Kappa (Quadratic): {kappa_quadratic:.4f}")


In [ ]:
# deepseek (kappa, quadratic kappa, confusion matrix)

import pandas as pd
from sklearn.metrics import cohen_kappa_score
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_excel("/kaggle/working/mistral_traduzioni_con_score.xlsx") 


print("\nStatistiche descrittive:")
print(df[['Human Score', 'Score']].describe())

# Calcola Cohen's Kappa
kappa = cohen_kappa_score(df['Human Score'], df['Score'])
weighted_kappa = cohen_kappa_score(df['Human Score'], df['Score'], weights='linear')

# Calcola l'accordo percentuale
agreement = np.mean(df['Human Score'] == df['Score'])

# Funzione per interpretare Kappa
def interpret_kappa(k):
    if k < 0:
        return "Accordo peggiore del caso casuale"
    elif k <= 0.2:
        return "Accordo trascurabile"
    elif k <= 0.4:
        return "Accordo debole"
    elif k <= 0.6:
        return "Accordo moderato"
    elif k <= 0.8:
        return "Accordo sostanziale"
    else:
        return "Accordo quasi perfetto"

# Risultati
print("\n" + "="*50)
print(f"Cohen's Kappa coefficient: {kappa:.3f}")
print(f"Weighted Cohen's Kappa: {weighted_kappa:.3f}")
print(f"Percentuale accordo: {agreement:.1%}")
print(f"Interpretazione: {interpret_kappa(kappa)}")
print("="*50)

# Matrice di confusione
conf_matrix = pd.crosstab(
    df['Human Score'], 
    df['Score'], 
    rownames=['Umano'],
    colnames=['LLM'],
    margins=True
)

print("\nMatrice di confusione:")
print(conf_matrix)

# Visualizzazione matrice di confusione
plt.figure(figsize=(10, 8))
sns.heatmap(
    conf_matrix.iloc[:-1, :-1],  # Escludi totali
    annot=True, 
    fmt='d',
    cmap='Blues',
    cbar_kws={'label': 'Numero di valutazioni'}
)
plt.title('Confronto tra Valutazioni Umane e LLM')
plt.xlabel('Valutazione LLM')
plt.ylabel('Valutazione Umana')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

# Analisi delle differenze
df['differenza'] = df['Human Score'] - df['Score']
df['differenza_assoluta'] = abs(df['differenza'])

print("\nAnalisi delle differenze:")
print(f"Differenza media: {df['differenza'].mean():.2f}")
print(f"Differenza assoluta media: {df['differenza_assoluta'].mean():.2f}")
print(f"Massima differenza: {df['differenza_assoluta'].max()}")

# Esempi con maggiori discrepanze
discrepanti = df[df['differenza_assoluta'] >= 2].sort_values('differenza_assoluta', ascending=False)
print(f"\nFrasi con maggiori discrepanze ({len(discrepanti)}):")
print(discrepanti[['frase_tradotta', 'valutazione_umana', 'valutazione_llm', 'differenza']])